In [ ]:
%pip install -U transformers datasets peft accelerate

In [ ]:
import sys
import torch
import transformers
import datasets
import peft
import accelerate

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)

print("\nCUDA available:", torch.cuda.is_available())
print("Device: CPU")

In [ ]:
from pathlib import Path
import json

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'backend').is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'backend').is_dir():
    raise FileNotFoundError('Could not find the HireAI project root. Run this notebook from within the project directory.')
DATASET_PROCESSED = (
    PROJECT_ROOT
    / "ai"
    / "datasets"
    / "processed"
)

V3_TRAIN = DATASET_PROCESSED / "hireai_train_v3.jsonl"
V3_VALIDATION = DATASET_PROCESSED / "hireai_validation_v3.jsonl"

print("Train exists:", V3_TRAIN.exists())
print("Validation exists:", V3_VALIDATION.exists())

In [ ]:
examples = []

with open(V3_TRAIN, "r", encoding="utf-8") as f:
    for _ in range(10):
        line = f.readline()

        if not line:
            break

        examples.append(json.loads(line))

print("Examples loaded:", len(examples))

In [ ]:
example = examples[0]

print("Instruction:")
print(example["instruction"])

print("\nInput:")
print(example["input"])

print("\nOutput:")
print(json.dumps(
    example["output"],
    indent=2,
    ensure_ascii=False
))

In [ ]:
def format_training_example(example):
    output_json = json.dumps(
        example["output"],
        ensure_ascii=False
    )

    return (
        "### Instruction:\n"
        f"{example['instruction']}\n\n"
        "### Resume:\n"
        f"{example['input']}\n\n"
        "### Response:\n"
        f"{output_json}"
    )


training_texts = [
    format_training_example(example)
    for example in examples
]

print(training_texts[0])

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_dict({
    "text": training_texts
})

print(train_dataset)

In [ ]:
MODEL_NAME = "Qwen/Qwen3-0.6B"

print("Training base model:", MODEL_NAME)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print("Tokenizer loaded successfully.")
print("Vocabulary size:", tokenizer.vocab_size)

In [ ]:
tokenized_full = tokenizer(
    training_texts,
    truncation=False
)

lengths = [
    len(ids)
    for ids in tokenized_full["input_ids"]
]

print("Token lengths:", lengths)
print("Maximum tokens:", max(lengths))
print("Average tokens:", sum(lengths) / len(lengths))
print("Minimum tokens:", min(lengths))

In [ ]:
tokenized_512 = tokenizer(
    training_texts,
    truncation=True,
    max_length=512
)

lengths_512 = [
    len(ids)
    for ids in tokenized_512["input_ids"]
]

truncated_count = sum(
    1 for length in lengths_512
    if length >= 512
)

print("Maximum after 512-token limit:", max(lengths_512))
print("Average after 512-token limit:", sum(lengths_512) / len(lengths_512))
print("Examples reaching 512 tokens:", truncated_count)

In [ ]:
tokenized_1024 = tokenizer(
    training_texts,
    truncation=True,
    max_length=1024
)

lengths_1024 = [
    len(ids)
    for ids in tokenized_1024["input_ids"]
]

truncated_count = sum(
    1 for length in lengths_1024
    if length >= 1024
)

print("Maximum after 1024-token limit:", max(lengths_1024))
print("Average after 1024-token limit:", sum(lengths_1024) / len(lengths_1024))
print("Examples reaching 1024 tokens:", truncated_count)

In [ ]:
max_index = lengths.index(max(lengths))

print("Longest example index:", max_index)
print("Longest example tokens:", lengths[max_index])

print("\nLongest example text:\n")
print(training_texts[max_index])

In [ ]:
from transformers import AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen3-0.6B"

print("Loading model:", MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME
)

print("Model loaded successfully.")

In [ ]:
total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(
    f"Total parameters: "
    f"{total_params / 1e6:.2f}M"
)

print(
    f"Trainable parameters: "
    f"{trainable_params / 1e6:.2f}M"
)

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
)

model = get_peft_model(
    model,
    lora_config
)

model.print_trainable_parameters()

In [ ]:
import psutil

memory = psutil.virtual_memory()

print(
    f"RAM used: {memory.used / (1024**3):.2f} GB"
)

print(
    f"RAM available: {memory.available / (1024**3):.2f} GB"
)

print(
    f"RAM usage: {memory.percent:.1f}%"
)

In [ ]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.config.pad_token_id = tokenizer.pad_token_id

print("Pad token:", tokenizer.pad_token)
print("Pad token ID:", tokenizer.pad_token_id)

In [ ]:
def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=1024,
        padding=False,
    )


tokenized_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

print(tokenized_dataset)

In [ ]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

print("Data collator created successfully.")

In [ ]:
from transformers import TrainingArguments

OUTPUT_DIR = PROJECT_ROOT / "ai" / "models" / "smoke_test"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),

    num_train_epochs=1,

    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,

    learning_rate=2e-4,
    weight_decay=0.0,

    logging_steps=1,
    save_strategy="no",

    report_to="none",

    fp16=False,
    bf16=False,

    dataloader_num_workers=0,

    optim="adamw_torch",

    remove_unused_columns=False,
)

print("Training configuration created.")

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

print("Trainer created successfully.")

In [ ]:
import psutil

memory = psutil.virtual_memory()

print(f"RAM used: {memory.used / (1024**3):.2f} GB")
print(f"RAM available: {memory.available / (1024**3):.2f} GB")
print(f"RAM usage: {memory.percent:.1f}%")

In [ ]:
import time

print("Starting HireAI LoRA smoke test...")
print("Training examples:", len(tokenized_dataset))
print("Device: CPU")

start_time = time.time()

train_result = trainer.train()

elapsed = time.time() - start_time

print("\nTraining completed.")
print(f"Training time: {elapsed / 60:.2f} minutes")
print(f"Training loss: {train_result.training_loss:.4f}")

In [ ]:
memory = psutil.virtual_memory()

print(f"RAM used: {memory.used / (1024**3):.2f} GB")
print(f"RAM available: {memory.available / (1024**3):.2f} GB")
print(f"RAM usage: {memory.percent:.1f}%")

In [ ]:
ADAPTER_DIR = (
    PROJECT_ROOT
    / "ai"
    / "models"
    / "hireai_qwen3_0.6b_lora_smoketest"
)

ADAPTER_DIR.mkdir(
    parents=True,
    exist_ok=True
)

trainer.model.save_pretrained(
    ADAPTER_DIR
)

tokenizer.save_pretrained(
    ADAPTER_DIR
)

print("LoRA adapter saved to:")
print(ADAPTER_DIR)

In [ ]:
model.eval()

print("Model is in evaluation mode.")

In [ ]:
test_example = examples[0]

prompt = (
    "### Instruction:\n"
    f"{test_example['instruction']}\n\n"
    "### Resume:\n"
    f"{test_example['input']}\n\n"
    "### Response:\n"
)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

print("Input tokens:", inputs["input_ids"].shape[1])

In [ ]:
import torch

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )

generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

generated_text = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print(generated_text)

In [ ]:
print("EXPECTED OUTPUT:")
print(
    json.dumps(
        test_example["output"],
        indent=2,
        ensure_ascii=False
    )
)

print("\nMODEL OUTPUT:")
print(generated_text)

In [ ]:
import json

V3_TEST = PROJECT_ROOT / "ai" / "datasets" / "processed" / "hireai_test_v3.jsonl"

with open(V3_TEST, "r", encoding="utf-8") as f:
    test_examples = [json.loads(line) for line in f]

print("Test examples:", len(test_examples))

unseen_example = test_examples[0]

print("\nUNSEEN RESUME:")
print(unseen_example["input"])

print("\nEXPECTED OUTPUT:")
print(json.dumps(
    unseen_example["output"],
    indent=2,
    ensure_ascii=False
))

In [ ]:
test_prompt = (
    "### Instruction:\n"
    f"{unseen_example['instruction']}\n\n"
    "### Resume:\n"
    f"{unseen_example['input']}\n\n"
    "### Response:\n"
)

test_inputs = tokenizer(
    test_prompt,
    return_tensors="pt"
)

print("Input tokens:", test_inputs["input_ids"].shape[1])

In [ ]:
with torch.no_grad():
    test_outputs = model.generate(
        **test_inputs,
        max_new_tokens=512,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        repetition_penalty=1.1,
    )

test_generated_tokens = test_outputs[0][
    test_inputs["input_ids"].shape[1]:
]

test_generated_text = tokenizer.decode(
    test_generated_tokens,
    skip_special_tokens=True
)

print(test_generated_text)

In [ ]:
def try_parse_json(text):
    text = text.strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None


parsed_test_output = try_parse_json(test_generated_text)

if parsed_test_output is None:
    print("❌ Invalid JSON")
else:
    print("✅ Valid JSON")
    print(json.dumps(
        parsed_test_output,
        indent=2,
        ensure_ascii=False
    ))

In [ ]:
import json

V3_DATASET = PROJECT_ROOT / "ai" / "datasets" / "processed" / "hireai_dataset_v3.jsonl"

with open(V3_DATASET, "r", encoding="utf-8") as f:
    v3_examples = [json.loads(line) for line in f]

print("Total examples:", len(v3_examples))

In [ ]:
REQUIRED_TOP_LEVEL_KEYS = {
    "candidate",
    "summary",
    "skills",
    "job_titles",
    "experience",
    "education",
    "projects",
    "certifications",
}

errors = []

for i, example in enumerate(v3_examples):

    output = example.get("output")

    if not isinstance(output, dict):
        errors.append((i, "output is not a dictionary"))
        continue

    # Check top-level schema
    missing = REQUIRED_TOP_LEVEL_KEYS - set(output.keys())

    if missing:
        errors.append((i, f"missing keys: {missing}"))

    # Candidate
    candidate = output.get("candidate")

    if not isinstance(candidate, dict):
        errors.append((i, "candidate is not a dictionary"))
    else:
        for key in ["name", "email", "phone"]:
            if key not in candidate:
                errors.append((i, f"candidate missing {key}"))

    # List fields
    for key in [
        "skills",
        "job_titles",
        "experience",
        "education",
        "projects",
        "certifications",
    ]:
        if not isinstance(output.get(key), list):
            errors.append((i, f"{key} is not a list"))

print("Schema errors:", len(errors))

if errors:
    for error in errors[:20]:
        print(error)
else:
    print("✅ All examples passed the basic schema check.")

In [ ]:
nested_errors = []

for i, example in enumerate(v3_examples):
    output = example["output"]

    for item in output["experience"]:
        required = {
            "company",
            "job_title",
            "start_date",
            "end_date",
            "description",
        }

        missing = required - set(item.keys())

        if missing:
            nested_errors.append(
                (i, "experience", missing, item)
            )

    for item in output["education"]:
        required = {
            "degree",
            "institution",
            "year",
        }

        missing = required - set(item.keys())

        if missing:
            nested_errors.append(
                (i, "education", missing, item)
            )

    for item in output["projects"]:
        required = {
            "name",
            "description",
        }

        missing = required - set(item.keys())

        if missing:
            nested_errors.append(
                (i, "projects", missing, item)
            )

print("Nested structure errors:", len(nested_errors))

for error in nested_errors[:20]:
    print(error)

HireAI Fine-Tuning Pipeline V2
Response-only LoRA training

In [ ]:
from pathlib import Path
import json
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)

from peft import (
    LoraConfig,
    get_peft_model,
)

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'backend').is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'backend').is_dir():
    raise FileNotFoundError('Could not find the HireAI project root. Run this notebook from within the project directory.')
DATASET_PROCESSED = (
    PROJECT_ROOT / "ai" / "datasets" / "processed"
)

V3_TRAIN = DATASET_PROCESSED / "hireai_train_v3.jsonl"
V3_VALIDATION = DATASET_PROCESSED / "hireai_validation_v3.jsonl"

MODEL_NAME = "Qwen/Qwen3-0.6B"

print("Model:", MODEL_NAME)
print("Train dataset exists:", V3_TRAIN.exists())
print("Validation dataset exists:", V3_VALIDATION.exists())

In [ ]:
with open(V3_TRAIN, "r", encoding="utf-8") as f:
    train_examples = [json.loads(line) for line in f]

with open(V3_VALIDATION, "r", encoding="utf-8") as f:
    validation_examples = [json.loads(line) for line in f]

print("Training examples:", len(train_examples))
print("Validation examples:", len(validation_examples))

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded.")
print("Vocabulary size:", tokenizer.vocab_size)
print("Pad token:", tokenizer.pad_token)

In [ ]:
SYSTEM_PROMPT = """You are HireAI's resume extraction model.

Extract structured information from the resume.

Return ONLY valid JSON using exactly this structure:

{
  "candidate": {
    "name": null,
    "email": null,
    "phone": null
  },
  "summary": null,
  "skills": [],
  "job_titles": [],
  "experience": [],
  "education": [],
  "projects": [],
  "certifications": []
}

Rules:

1. Extract only information supported by the resume.
2. Do not invent missing information.
3. Use null for missing scalar values.
4. Use [] for missing list values.
5. job_titles must contain only actual job titles.
6. Each experience item must contain company, job_title, start_date, end_date, and description.
7. Each education item must contain degree, institution, and year.
8. Each project must contain name and description.
9. Preserve the meaning of the original resume.
10. Return JSON only."""

In [ ]:
def build_messages(example):
    return [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": (
                "Extract the structured information from this resume:\n\n"
                + example["input"]
            ),
        },
        {
            "role": "assistant",
            "content": json.dumps(
                example["output"],
                ensure_ascii=False,
                separators=(",", ":"),
            ),
        },
    ]

In [ ]:
messages = build_messages(train_examples[0])

for message in messages:
    print("\nROLE:", message["role"])
    print(message["content"][:500])

In [ ]:
formatted_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=False,
)

print(formatted_prompt[:4000])

In [ ]:
lengths = []

for example in train_examples[:100]:
    messages = build_messages(example)

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    token_count = len(
        tokenizer(
            text,
            add_special_tokens=False
        )["input_ids"]
    )

    lengths.append(token_count)

print("Samples checked:", len(lengths))
print("Maximum:", max(lengths))
print("Average:", sum(lengths) / len(lengths))
print("Minimum:", min(lengths))

In [ ]:
from pathlib import Path
import json
import torch

from transformers import AutoTokenizer

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'backend').is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'backend').is_dir():
    raise FileNotFoundError('Could not find the HireAI project root. Run this notebook from within the project directory.')
DATASET_PROCESSED = (
    PROJECT_ROOT / "ai" / "datasets" / "processed"
)

V3_TRAIN = DATASET_PROCESSED / "hireai_train_v3.jsonl"
V3_VALIDATION = DATASET_PROCESSED / "hireai_validation_v3.jsonl"

MODEL_NAME = "Qwen/Qwen3-0.6B"

print("Project:", PROJECT_ROOT)
print("Train exists:", V3_TRAIN.exists())
print("Validation exists:", V3_VALIDATION.exists())

In [ ]:
with open(V3_TRAIN, "r", encoding="utf-8") as f:
    train_examples = [json.loads(line) for line in f]

with open(V3_VALIDATION, "r", encoding="utf-8") as f:
    validation_examples = [json.loads(line) for line in f]

print("Training examples:", len(train_examples))
print("Validation examples:", len(validation_examples))

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded.")
print("Vocabulary size:", tokenizer.vocab_size)
print("Pad token:", tokenizer.pad_token)

In [ ]:
SYSTEM_PROMPT = """You are HireAI's resume extraction model.

Extract structured information from the resume.

Return ONLY valid JSON using exactly this structure:

{
  "candidate": {
    "name": null,
    "email": null,
    "phone": null
  },
  "summary": null,
  "skills": [],
  "job_titles": [],
  "experience": [],
  "education": [],
  "projects": [],
  "certifications": []
}

Rules:

1. Extract only information supported by the resume.
2. Do not invent missing information.
3. Use null for missing scalar values.
4. Use [] for missing list values.
5. job_titles must contain only actual job titles.
6. Each experience item must contain company, job_title, start_date, end_date, and description.
7. Each education item must contain degree, institution, and year.
8. Each project must contain name and description.
9. Preserve the meaning of the original resume.
10. Return JSON only."""

In [ ]:
def build_prompt_and_response(example):
    messages_without_response = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": (
                "Extract the structured information from this resume:\n\n"
                + example["input"]
            ),
        },
    ]

    response = json.dumps(
        example["output"],
        ensure_ascii=False,
        separators=(",", ":"),
    )

    prompt_text = tokenizer.apply_chat_template(
        messages_without_response,
        tokenize=False,
        add_generation_prompt=True,
    )

    return prompt_text, response

In [ ]:
prompt_text, response_text = build_prompt_and_response(
    train_examples[0]
)

print("===== PROMPT =====")
print(prompt_text)

print("\n===== RESPONSE =====")
print(response_text)

In [ ]:
MAX_LENGTH = 1024


def tokenize_for_training(example):
    prompt_text, response_text = build_prompt_and_response(example)

    prompt_tokens = tokenizer(
        prompt_text,
        add_special_tokens=False,
    )

    response_tokens = tokenizer(
        response_text,
        add_special_tokens=False,
    )

    input_ids = (
        prompt_tokens["input_ids"]
        + response_tokens["input_ids"]
        + [tokenizer.eos_token_id]
    )

    attention_mask = [1] * len(input_ids)

    # Ignore prompt tokens when calculating loss
    labels = (
        [-100] * len(prompt_tokens["input_ids"])
        + response_tokens["input_ids"]
        + [tokenizer.eos_token_id]
    )

    # Safety check
    if len(input_ids) > MAX_LENGTH:
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

In [ ]:
test_tokens = tokenize_for_training(train_examples[0])

print("Total tokens:", len(test_tokens["input_ids"]))

print(
    "Ignored tokens:",
    test_tokens["labels"].count(-100)
)

print(
    "Training tokens:",
    sum(
        1 for x in test_tokens["labels"]
        if x != -100
    )
)

In [ ]:
prompt_length = test_tokens["labels"].count(-100)
response_length = len(test_tokens["labels"]) - prompt_length

print("Prompt tokens:", prompt_length)
print("Response tokens:", response_length)
print("Total:", len(test_tokens["labels"]))

In [ ]:
response_lengths = []

for example in train_examples[:100]:
    _, response_text = build_prompt_and_response(example)

    response_token_count = len(
        tokenizer(
            response_text,
            add_special_tokens=False
        )["input_ids"]
    )

    response_lengths.append(response_token_count)

print("Response maximum:", max(response_lengths))
print(
    "Response average:",
    sum(response_lengths) / len(response_lengths)
)
print("Response minimum:", min(response_lengths))

In [ ]:
from dataclasses import dataclass
from typing import Any

import torch


@dataclass
class HireAIDataCollator:
    tokenizer: Any
    label_pad_token_id: int = -100

    def __call__(self, features):
        max_length = max(
            len(feature["input_ids"])
            for feature in features
        )

        batch_input_ids = []
        batch_attention_mask = []
        batch_labels = []

        for feature in features:
            padding_length = (
                max_length - len(feature["input_ids"])
            )

            batch_input_ids.append(
                feature["input_ids"]
                + [self.tokenizer.pad_token_id] * padding_length
            )

            batch_attention_mask.append(
                feature["attention_mask"]
                + [0] * padding_length
            )

            batch_labels.append(
                feature["labels"]
                + [self.label_pad_token_id] * padding_length
            )

        return {
            "input_ids": torch.tensor(
                batch_input_ids,
                dtype=torch.long,
            ),
            "attention_mask": torch.tensor(
                batch_attention_mask,
                dtype=torch.long,
            ),
            "labels": torch.tensor(
                batch_labels,
                dtype=torch.long,
            ),
        }

In [ ]:
smoke_examples = train_examples[:10]

smoke_tokens = [
    tokenize_for_training(example)
    for example in smoke_examples
]

print("Smoke-test examples:", len(smoke_tokens))

In [ ]:
collator = HireAIDataCollator(
    tokenizer=tokenizer
)

batch = collator(smoke_tokens[:2])

print("Input shape:", batch["input_ids"].shape)
print("Labels shape:", batch["labels"].shape)

print(
    "Ignored label count:",
    (batch["labels"] == -100).sum().item()
)

In [ ]:
from datasets import Dataset

smoke_dataset = Dataset.from_list(
    smoke_tokens
)

print(smoke_dataset)
print(smoke_dataset.column_names)

In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME
)

model.config.pad_token_id = tokenizer.pad_token_id

print("Model loaded.")
print(
    "Parameters:",
    sum(p.numel() for p in model.parameters()) / 1e6,
    "M"
)

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
)

model = get_peft_model(
    model,
    lora_config
)

model.print_trainable_parameters()

In [ ]:
from dataclasses import dataclass
from typing import Any

import torch


@dataclass
class HireAIDataCollator:
    tokenizer: Any
    label_pad_token_id: int = -100

    def __call__(self, features):
        max_length = max(
            len(feature["input_ids"])
            for feature in features
        )

        batch_input_ids = []
        batch_attention_mask = []
        batch_labels = []

        for feature in features:
            padding_length = (
                max_length - len(feature["input_ids"])
            )

            batch_input_ids.append(
                feature["input_ids"]
                + [self.tokenizer.pad_token_id] * padding_length
            )

            batch_attention_mask.append(
                feature["attention_mask"]
                + [0] * padding_length
            )

            batch_labels.append(
                feature["labels"]
                + [self.label_pad_token_id] * padding_length
            )

        return {
            "input_ids": torch.tensor(
                batch_input_ids,
                dtype=torch.long,
            ),
            "attention_mask": torch.tensor(
                batch_attention_mask,
                dtype=torch.long,
            ),
            "labels": torch.tensor(
                batch_labels,
                dtype=torch.long,
            ),
        }


collator = HireAIDataCollator(
    tokenizer=tokenizer
)

In [ ]:
batch = collator(
    smoke_tokens[:2]
)

print("Input shape:", batch["input_ids"].shape)
print("Attention shape:", batch["attention_mask"].shape)
print("Labels shape:", batch["labels"].shape)

print(
    "Ignored labels:",
    (batch["labels"] == -100).sum().item()
)

print(
    "Training labels:",
    (batch["labels"] != -100).sum().item()
)

In [ ]:
labels = batch["labels"]

for i in range(labels.shape[0]):
    ignored = (labels[i] == -100).sum().item()
    trainable = (labels[i] != -100).sum().item()

    print(
        f"Example {i}: "
        f"ignored={ignored}, "
        f"trainable={trainable}"
    )

In [ ]:
model.train()

batch = {
    key: value.to("cpu")
    for key, value in batch.items()
}

outputs = model(**batch)

print("Loss:", outputs.loss.item())

In [ ]:
outputs.loss.backward()

print("Backward pass completed.")

In [ ]:
gradient_count = 0
gradient_parameters = 0

for name, parameter in model.named_parameters():
    if parameter.requires_grad and parameter.grad is not None:
        gradient_count += 1
        gradient_parameters += parameter.grad.numel()

print("Parameters with gradients:", gradient_count)
print("Gradient values:", gradient_parameters)

In [ ]:
model.zero_grad(set_to_none=True)
model.eval()

print("Model gradients cleared.")
print("Model is in evaluation mode.")

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME
)

base_model.config.pad_token_id = tokenizer.pad_token_id
base_model.eval()

print("Base model loaded.")

In [ ]:
def generate_base_output(example, max_new_tokens=512):
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": (
                "Extract the structured information from this resume:\n\n"
                + example["input"]
            ),
        },
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    )

    with torch.no_grad():
        outputs = base_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
        )

    generated_tokens = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    )

In [ ]:
baseline_output = generate_base_output(
    validation_examples[0]
)

print(baseline_output)

In [ ]:
def extract_json_object(text):
    text = text.strip()

    # Remove Qwen reasoning markers if present
    if "<think>" in text and "</think>" in text:
        text = text.split("</think>", 1)[1].strip()

    # Remove markdown fences
    if text.startswith("```"):
        lines = text.splitlines()

        if lines and lines[0].startswith("```"):
            lines = lines[1:]

        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]

        text = "\n".join(lines).strip()

    # Find the JSON object
    start = text.find("{")
    end = text.rfind("}")

    if start == -1 or end == -1 or end <= start:
        return None

    json_text = text[start:end + 1]

    try:
        return json.loads(json_text)
    except json.JSONDecodeError:
        return None

In [ ]:
parsed_baseline = extract_json_object(
    baseline_output
)

if parsed_baseline is None:
    print("❌ Invalid JSON")
else:
    print("✅ Valid JSON")
    print(json.dumps(
        parsed_baseline,
        indent=2,
        ensure_ascii=False
    ))

In [ ]:
def generate_base_output_fast(example, max_new_tokens=160):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": (
            "Extract the structured information from this resume:\n\n"
            + example["input"]
        )},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    with torch.no_grad():
        outputs = base_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
        )

    generated_tokens = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

In [ ]:
import time

start = time.time()

baseline_output = generate_base_output_fast(
    validation_examples[0],
    max_new_tokens=160
)

elapsed = time.time() - start

print(f"Time: {elapsed:.1f} seconds")
print("=" * 80)
print(baseline_output)

In [ ]:
parsed_baseline = extract_json_object(baseline_output)

if parsed_baseline is None:
    print("\n❌ Invalid JSON")
else:
    print("\n✅ Valid JSON")
    print(json.dumps(
        parsed_baseline,
        indent=2,
        ensure_ascii=False
    ))

In [ ]:
import gc
import torch

# Remove the base model used for slow inference
if "base_model" in globals():
    del base_model

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Memory cleanup completed.")

In [ ]:
print(tokenizer.chat_template)

In [ ]:
test_messages = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT
    },
    {
        "role": "user",
        "content": "Extract the structured information from this resume:\n\n"
                   + validation_examples[0]["input"]
    }
]

test_prompt = tokenizer.apply_chat_template(
    test_messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

print(test_prompt[-2000:])

In [ ]:
def build_prompt_and_response(example):
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": (
                "Extract the structured information from this resume:\n\n"
                + example["input"]
            )
        },
    ]

    response = json.dumps(
        example["output"],
        ensure_ascii=False,
        separators=(",", ":"),
    )

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    return prompt_text, response

In [ ]:
prompt_text, response_text = build_prompt_and_response(
    train_examples[0]
)

print("PROMPT")
print("=" * 80)
print(prompt_text[-1500:])

print("\nTARGET RESPONSE")
print("=" * 80)
print(response_text)

In [ ]:
MAX_LENGTH = 1024

def tokenize_for_training(example):
    prompt_text, response_text = build_prompt_and_response(example)

    prompt_tokens = tokenizer(
        prompt_text,
        add_special_tokens=False,
    )

    response_tokens = tokenizer(
        response_text,
        add_special_tokens=False,
    )

    input_ids = (
        prompt_tokens["input_ids"]
        + response_tokens["input_ids"]
        + [tokenizer.eos_token_id]
    )

    attention_mask = [1] * len(input_ids)

    labels = (
        [-100] * len(prompt_tokens["input_ids"])
        + response_tokens["input_ids"]
        + [tokenizer.eos_token_id]
    )

    if len(input_ids) > MAX_LENGTH:
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_list([
    tokenize_for_training(example)
    for example in train_examples
])

validation_dataset = Dataset.from_list([
    tokenize_for_training(example)
    for example in validation_examples
])

print("Train examples:", len(train_dataset))
print("Validation examples:", len(validation_dataset))
print("First train example tokens:", len(train_dataset[0]["input_ids"]))

In [ ]:
train_lengths = [
    len(item["input_ids"])
    for item in train_dataset
]

validation_lengths = [
    len(item["input_ids"])
    for item in validation_dataset
]

print("Train:")
print("  Max:", max(train_lengths))
print("  Avg:", sum(train_lengths) / len(train_lengths))
print("  >1024:", sum(x > 1024 for x in train_lengths))

print("\nValidation:")
print("  Max:", max(validation_lengths))
print("  Avg:", sum(validation_lengths) / len(validation_lengths))
print("  >1024:", sum(x > 1024 for x in validation_lengths))

In [ ]:
print("Training dataset:", len(train_dataset))
print("Validation dataset:", len(validation_dataset))

print("\nFirst example:")
print("Total tokens:", len(train_dataset[0]["input_ids"]))
print(
    "Prompt tokens:",
    sum(x == -100 for x in train_dataset[0]["labels"])
)
print(
    "Response tokens:",
    sum(x != -100 for x in train_dataset[0]["labels"])
)

print("\nEOS token ID:", tokenizer.eos_token_id)
print("PAD token ID:", tokenizer.pad_token_id)

In [ ]:
import gc
import torch

if "model" in globals():
    del model

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Previous model cleared.")

In [ ]:
from transformers import AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen3-0.6B"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME
)

model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False

print(f"Model loaded: {MODEL_NAME}")
print(
    f"Parameters: "
    f"{sum(p.numel() for p in model.parameters()):,}"
)

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
)

model = get_peft_model(
    model,
    lora_config
)

model.print_trainable_parameters()

In [ ]:
print(train_dataset)
print(validation_dataset)

In [ ]:
data_collator = HireAIDataCollator(
    tokenizer=tokenizer
)

print("HireAI data collator ready.")

In [ ]:
from transformers import TrainingArguments

OUTPUT_DIR = PROJECT_ROOT / "ai" / "models" / "hireai_qwen3_0.6b_lora"

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),

    # Training
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,

    # Validation
    per_device_eval_batch_size=2,
    eval_strategy="epoch",

    # Optimization
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=100,

    # Logging / checkpoints
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,

    # Performance
    fp16=False,
    bf16=False,
    dataloader_num_workers=0,

    # Optimizer
    optim="adamw_torch",

    # Keep our custom labels
    remove_unused_columns=False,

    # No external tracking
    report_to="none",

    # Reproducibility
    seed=42,
)

print("Training arguments created successfully.")
print(training_args)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    data_collator=data_collator,
)

print("Trainer created successfully.")

In [ ]:
print("Trainer:", type(trainer).__name__)
print("Training examples:", len(trainer.train_dataset))
print("Validation examples:", len(trainer.eval_dataset))

print("\nTraining configuration:")
print("Epochs:", trainer.args.num_train_epochs)
print("Train batch size:", trainer.args.per_device_train_batch_size)
print("Gradient accumulation:", trainer.args.gradient_accumulation_steps)
print("Learning rate:", trainer.args.learning_rate)
print("Warmup steps:", trainer.args.warmup_steps)
print("Evaluation strategy:", trainer.args.eval_strategy)
print("Save strategy:", trainer.args.save_strategy)

In [ ]:
batch = next(iter(trainer.get_train_dataloader()))

print("Batch keys:")
print(batch.keys())

print("\nShapes:")
for key, value in batch.items():
    print(f"{key}: {value.shape}")

print("\nLabel statistics:")

labels = batch["labels"]

ignored_tokens = (labels == -100).sum().item()
training_tokens = (labels != -100).sum().item()

print("Ignored tokens:", ignored_tokens)
print("Training tokens:", training_tokens)

In [ ]:
model.train()

batch = {
    key: value.to(model.device)
    for key, value in batch.items()
}

outputs = model(**batch)

print("Loss:", outputs.loss.item())
print("Forward pass: ✅")